# Exp 8: recruitment onset and time-resolved sharing with Control

Compare each condition only with its same-side control: **LeB** on the left and **RiB** on the right. No raster is needed. For every neuron with a qualifying control response, compare its first sustained onset in the control with its onset in each other condition, preserving fish and retained-neuron identity.

Both one-position and five-position flicker conditions are included: this analysis compares response timing and does not require a single spatial position. Run this notebook independently of notebooks 01/02.

In [ ]:
# Setup
from importlib import reload
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src.data_loading as data_loading
import src.several_fish_loading as several_loading
import src.active_neuron_analysis as active_analysis
import src.plotting_onset_comparison as onset_plot
active_analysis = reload(active_analysis)
onset_plot = reload(onset_plot)

## 1. Cohort and conditions

The cohort, four stimulus blocks and condition order match Exp 8 notebook 01. Paths use the local Lab root, including `SOCIAL_FILTERS_LAB_ROOT` when configured. The loader requires every requested fish and stimulus.

In [ ]:
USER_SETTINGS = {
    'experiment_name': 'Exp_8_accumulation_ev',
    'lab_root': data_loading.resolve_lab_root(),
    'fish_ids': ['L800_f02', 'L800_f03', 'L800_f09', 'L800_f04',
                 'L800_f05', 'L800_f07', 'L800_f10', 'L800_f11'],
    'selected_blocks': ['B1', 'B2', 'B3', 'B4'],
    'verbose_loading': False,
    'timing': {'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 32.0},
    'stim_order': [5, 4, 3, 2, 1, 8, 7, 6, 16, 15, 14, 13, 12, 11, 10, 9],
}
SIDE_CONDITIONS = {
    'left': {'control': 'LeB', 'stimuli':
             ['LeB', 'LeBseg2', 'LeBseg1', 'LeBseg1rnd', 'LeBseg135', 'LeRock', 'Leflick5', 'Leflick1']},
    'right': {'control': 'RiB', 'stimuli':
              ['RiB', 'RiBseg2', 'RiBseg1', 'RiBseg1rnd', 'RiBseg135', 'RiBrock', 'RiBflick5', 'RiBflick1']},
}

In [ ]:
raster_inputs = several_loading.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
all_fish_data = raster_inputs['all_fish_data']
fish_ids = raster_inputs['fish_ids']
timing = raster_inputs['timing']
print(f'Loaded {len(fish_ids)} fish; four repetitions per condition are required below.')

## 2. Define and prepare onset comparisons

**Time zero is the common 8 s stimulus-program start**, matching notebook 01's Cell 06 timing. It is not each condition's first coordinate/radius change (8.3, 8.6 or 9.2 s). The default onset window is **[0, 9) s**, ending at the 17 s program end. Thus delays relative to the common program remain visible. Edit the settings below to change the window/bin width; a final shorter bin is allowed. All bins include their left edge and exclude their right edge.

First apply Cell 06's existing activity rule: at least 10% significant frames and a two-frame epoch in at least two of four trials, in its motion-duration-plus-12 s response window. Then detect onset inside the narrower onset window: the first of **two consecutive imaging frames** with at least **50% of trials significant at each frame**. These frames can be supported by different trials; this is a consensus-trace onset, not a median single-trial latency. An epoch must fit wholly within the onset window. Activity and onset settings are separate and explicit.

The reference population is **control-active neurons with a qualifying control onset inside the window**. A comparison onset also requires that condition to pass the activity rule. The audit records control-active neurons excluded for lacking an onset in this window. No detected onset does not prove the neuron never responds: its response may be outside the window or fail the criteria. Onsets already present at time zero are assigned zero; this does not establish that their activity began after time zero.

In [ ]:
ONSET_WINDOW_S = (0.0, 9.0)
BIN_WIDTH_S = 1.0  # Try 1.5 or 2.0 to check sensitivity to bin boundaries.
ONSET_SETTINGS = {
    'motion_onset_s': 8.0,
    'persistence_frames': 2,
    'min_significant_trial_fraction': 0.50,
    'active_fraction_threshold': 0.10,
    'min_epoch_s': 1.0,
    'min_active_reps': 2,
    'expected_reps': 4,
    'tau_s': 6.0,
}
if not np.isfinite(BIN_WIDTH_S) or BIN_WIDTH_S <= 0 or not 0 <= ONSET_WINDOW_S[0] < ONSET_WINDOW_S[1]:
    raise ValueError('Use a positive bin width and an increasing, nonnegative onset window.')
bin_edges_s = np.r_[np.arange(*ONSET_WINDOW_S, BIN_WIDTH_S), ONSET_WINDOW_S[1]]
onset_result = active_analysis.build_control_onset_comparison(
    all_fish_data, fish_ids, SIDE_CONDITIONS, bin_edges_s,
    fps_2p=timing['fps_2p'], t_pre_s=timing['t_pre_s'], **ONSET_SETTINGS,
)
display(onset_result['audit_table'])

## 3. Onset comparison heatmaps

For each stimulus, **rows are control-onset bins** and **columns are that stimulus's onset bins**. Each row divides by all reference neurons in that control bin **within each fish**, then averages the percentages with equal fish weight. A white box marks the same-bin diagonal; cells to its left/right represent earlier/later recruitment. The final column includes comparison-inactive neurons and neurons without a qualifying onset. A gray row has no eligible fish, rather than 0% sharing. The control-versus-self panel is diagonal by construction, not a measure of repeat reliability.

Each nonempty row sums to 100%, including the final column. All eight conditions retain the configured order. Figure controls below redraw cached results without recalculating onsets.

In [ ]:
HEATMAP_FIGSIZE = (16, 8)
HEATMAP_FONT_PT = 8
HEATMAP_CMAP = 'viridis'
heatmap_figures = {
    side: onset_plot.plot_control_onset_heatmaps(
        onset_result, side, figsize=HEATMAP_FIGSIZE, cmap=HEATMAP_CMAP,
        font_size=HEATMAP_FONT_PT,
    ) for side in SIDE_CONDITIONS
}
for plotted in heatmap_figures.values():
    display(plotted['figure'])
    plt.close(plotted['figure'])

## 4. How much sharing preserves recruitment time?

**Top:** percentage of control-bin neurons also recruited in the same onset bin. **Middle:** percentage with a qualifying onset anywhere in the analysis window. Their difference is the fraction recruited at a shifted time. These curves measure first onset, not continuing activity inside a bin. **Bottom:** reference-neuron counts per fish (gray) and their mean (black), showing when late bins have little support.

Thin colored lines show individual fish; thick curves average fish equally, excluding empty reference bins. The tables below retain the number of contributing fish and the exact raw counts. Same-bin agreement depends on bin boundaries, so compare wider bins before interpreting small changes. These are descriptive calcium-response onset comparisons; they do not establish precise spike latency, spatial selectivity or an independently validated temporal code.

In [ ]:
SUMMARY_FIGSIZE = (12, 7)
SUMMARY_FONT_PT = 9
SHOW_INDIVIDUAL_FISH = True
summary_figures = {
    side: onset_plot.plot_control_onset_summary(
        onset_result, side, figsize=SUMMARY_FIGSIZE, font_size=SUMMARY_FONT_PT,
        show_fish=SHOW_INDIVIDUAL_FISH,
    ) for side in SIDE_CONDITIONS
}
for plotted in summary_figures.values():
    display(plotted['figure'])
    plt.close(plotted['figure'])

## 5. Inspect exact numbers

Select one side, condition and control-onset bin below. `n_shared = n_earlier + n_same + n_later`; adding `n_no_onset` gives `n_control`. Counts identify neurons shared with the control, not neurons newly recruited outside the control population.

`onset_result['neuron_table']` preserves fish ID, original neuron ID, retained row, control/stimulus onset seconds and bin assignments. `fish_transitions` contains every onset-bin pair, with stimulus bin -1 meaning no detected onset. `bin_summary` and `transition_summary` hold equal-fish fractions and valid fish counts. All results remain in memory; this notebook writes no Lab files.

In [ ]:
TABLE_SIDE = 'right'
TABLE_STIMULUS = 'RiBseg1'
TABLE_CONTROL_BIN = 2  # Zero-based: bin 2 is [2, 3) s with the default settings.
fish_bin_counts = onset_result['fish_bins']
display(fish_bin_counts.loc[
    (fish_bin_counts['side'] == TABLE_SIDE)
    & (fish_bin_counts['stimulus'] == TABLE_STIMULUS)
    & (fish_bin_counts['control_bin'] == TABLE_CONTROL_BIN),
    ['fish_id', 'n_control', 'n_shared', 'n_earlier', 'n_same', 'n_later', 'n_no_onset'],
])

## 6. How much significant activity is shared with Control at matching times?

For each fish and **the same time bin in both conditions**, count significant neurons shared with same-side Control. The percentage is **100 × shared neurons / Control-active neurons in that bin**. A neuron can contribute in several bins. This measures continuing activity, separately from sections 3–4's first-onset comparisons. Stimulus-only neurons are counted in the tables but do not reduce this directional percentage.

Time zero remains the common program start (8 s after appearance). The default **[0, 14) s** window includes the 9 s program and 5 s afterwards, matching notebook 04's broad display span. Each bin includes its left edge and excludes its right edge. A neuron is active if, **within the bin**, it has a two-frame significant epoch (1 s at 2 Hz) and at least 10% significant frames **in at least two of four trials**. Each passing trial must contain the complete epoch; this is not a consensus-trace or first-onset decision. No whole-response eligibility gate is applied. Try 2 s bins to check sensitivity; epochs must fit wholly within each bin.

The four comparisons and colors match notebook 04's current distance display: **Seg 2, Seg 1, Seg 1/3/5 and one-position flicker**. Percentages are computed within fish, then averaged equally over fish with nonempty Control bins. Empty Control bins are undefined, not 0%; valid-fish counts and raw neuron counts remain available below. The lower plot shows Control support (individual fish in gray, mean in black).

Curves and SEM are descriptive. A fall in overlap can identify a period of reduced sharing, but does not by itself establish a statistical separation time. Control-self would be 100% by construction and is omitted; independent Control repetitions would be needed for a repeatability benchmark. Significant-trace detection and population separation are different meanings of significance.

In [ ]:
BIN_OVERLAP_WINDOW_S = (0.0, 14.0)
BIN_OVERLAP_WIDTH_S = 1.0  # Also inspect 2 s bins.
BIN_OVERLAP_SETTINGS = {
    'program_onset_s': 8.0, 'min_epoch_s': 1.0,
    'min_active_reps': 2, 'expected_reps': 4, 'active_fraction_threshold': 0.10,
}
BIN_OVERLAP_CONDITIONS = {
    'left': {'control': 'LeB', 'stimuli': ['LeB', 'LeBseg2', 'LeBseg1', 'LeBseg135', 'Leflick1']},
    'right': {'control': 'RiB', 'stimuli': ['RiB', 'RiBseg2', 'RiBseg1', 'RiBseg135', 'RiBflick1']},
}
if (not np.isfinite(BIN_OVERLAP_WIDTH_S) or BIN_OVERLAP_WIDTH_S <= 0
        or not np.isfinite(BIN_OVERLAP_WINDOW_S).all()
        or not 0 <= BIN_OVERLAP_WINDOW_S[0] < BIN_OVERLAP_WINDOW_S[1]):
    raise ValueError('Use a positive bin width and an increasing, nonnegative window.')
overlap_edges_s = np.r_[np.arange(*BIN_OVERLAP_WINDOW_S, BIN_OVERLAP_WIDTH_S), BIN_OVERLAP_WINDOW_S[1]]
bin_overlap_result = active_analysis.build_control_bin_overlap(
    all_fish_data, fish_ids, BIN_OVERLAP_CONDITIONS, overlap_edges_s,
    fps_2p=timing['fps_2p'], t_pre_s=timing['t_pre_s'], **BIN_OVERLAP_SETTINGS,
)

### Draw the selected comparisons

Redraw these cached results to change appearance without recomputing activity decisions. Shading is descriptive SEM across eligible fish, omitted where fewer than two fish contribute. The dotted line marks program end at 9 s.

In [ ]:
BIN_OVERLAP_FIGSIZE, BIN_OVERLAP_FONT_PT = (9, 5.5), 9
SHOW_BIN_OVERLAP_SEM, SHOW_BIN_OVERLAP_FISH = True, False
BIN_OVERLAP_COLORS = {
    'LeBseg2': '#4F947F', 'RiBseg2': '#4F947F',
    'LeBseg1': '#BE7844', 'RiBseg1': '#BE7844',
    'LeBseg135': '#8C6BB1', 'RiBseg135': '#8C6BB1',
    'Leflick1': '#569BB9', 'RiBflick1': '#569BB9',
}
BIN_OVERLAP_LABELS = {
    'LeBseg2': 'Seg 2', 'RiBseg2': 'Seg 2', 'LeBseg1': 'Seg 1', 'RiBseg1': 'Seg 1',
    'LeBseg135': 'Seg 1/3/5', 'RiBseg135': 'Seg 1/3/5',
    'Leflick1': 'One-position flicker', 'RiBflick1': 'One-position flicker',
}
bin_overlap_figures = {
    side: onset_plot.plot_control_bin_overlap(
        bin_overlap_result, side, colors=BIN_OVERLAP_COLORS, label_map=BIN_OVERLAP_LABELS,
        figsize=BIN_OVERLAP_FIGSIZE, font_size=BIN_OVERLAP_FONT_PT,
        show_sem=SHOW_BIN_OVERLAP_SEM, show_fish=SHOW_BIN_OVERLAP_FISH, program_end_s=9.0,
    ) for side in BIN_OVERLAP_CONDITIONS
}
for plotted in bin_overlap_figures.values():
    display(plotted['figure'])
    plt.close(plotted['figure'])

### Inspect support and exact numbers

Choose a side and comparison below. `n_control`, `n_stimulus`, `n_shared`, `n_control_only` and `n_stimulus_only` are exact per-fish counts. The summary reports mean sharing as a fraction, SEM, contributing fish and summed counts; **the displayed mean is not a pooled count ratio**. `bin_activity[side][fish_id][label]` has retained-neuron rows × bins, aligned to `neuron_ids[fish_id]`, for checking exact shared identities. `bin_windows` records the actual native samples used. These results remain in memory; no Lab files are written.

In [ ]:
BIN_COUNT_SIDE, BIN_COUNT_STIMULUS = 'left', 'LeBseg2'
display(bin_overlap_result['bin_summary'].query('side == @BIN_COUNT_SIDE and stimulus == @BIN_COUNT_STIMULUS'))
# Exact per-fish counts for the same selection:
# display(bin_overlap_result['fish_bins'].query('side == @BIN_COUNT_SIDE and stimulus == @BIN_COUNT_STIMULUS'))